#Práctica 2.2: Adquisición de Datos Híbrida e Integración (ETL)



In [ ]:
#Librerías
import sqlite3
import pandas as pd
import requests
from bs4 import BeautifulSoup
import pandas as pd




Fase 1: Simulación de Fuentes Locales (SQL y CSV).

In [ ]:
import sqlite3
import pandas as pd
import os

# --- PASO A: Definir la ruta de tu carpeta en Drive ---
# Asegúrate de que el nombre coincida con la carpeta que creaste
ruta_drive = '/content/drive/MyDrive/practica/'

# Creamos la carpeta si no existe (por seguridad)
if not os.path.exists(ruta_drive):
    os.makedirs(ruta_drive)

# --- PASO B: Guardar la Base de Datos SQL en Drive ---
conn = sqlite3.connect(ruta_drive + 'tarifas.db')
cursor = conn.cursor()
cursor.execute('''CREATE TABLE IF NOT EXISTS aduana (rating_id INTEGER, tarifa_gbp REAL)''')
cursor.executemany("INSERT INTO aduana VALUES (?, ?)", [(1, 5.0), (2, 4.0), (3, 3.0), (4, 2.0), (5, 1.0)])
conn.commit()
conn.close() # Es importante cerrar la conexión para que el archivo se escriba bien

# --- PASO C: Guardar el CSV en Drive ---
df_desc = pd.DataFrame({'tienda_id': [5], 'factor_descuento': [0.85]})
df_desc.to_csv(ruta_drive + 'descuentos.csv', index=False)

print(f"Archivos guardados en: {ruta_drive}")

Archivos guardados en: /content/drive/MyDrive/practica/


In [ ]:
# 1. Generar Base de Datos SQL (Tarifas de importación por calificación / estrellas)
conn = sqlite3.connect('tarifas.db') # Se corrigieron las comillas aquí
cursor = conn.cursor()
cursor.execute('''CREATE TABLE IF NOT EXISTS aduana (rating_id INTEGER, tarifa_gbp REAL)''')
cursor.executemany("INSERT INTO aduana VALUES (?, ?)", [(1, 5.0), (2, 4.0), (3, 3.0), (4, 2.0), (5, 1.0)])
conn.commit()

# 2. Generar Archivo Plano CSV (Descuento de la tienda)
df_desc = pd.DataFrame({'tienda_id': [5], 'factor_descuento': [0.85]}) # 15% de descuento
df_desc.to_csv('descuentos.csv', index=False) # Se corrigieron las comillas aquí

print("Base de datos y CSV generados correctamente.")

Base de datos y CSV generados correctamente.


Fase 2: Web Scraping Dinámico.

In [ ]:
# 1. Target URL 9)
url_scrape = "http://books.toscrape.com/catalogue/page-9.html"

#html_crudo = respuesta.text

# 2. Descargar el contenido
respuesta_web = requests.get(url_scrape)
html_crudo = respuesta_web.text
print(f"Tipo de 'html_crudo': {type(html_crudo)}")
# 3. Objeto parseado del DOM
sopa = BeautifulSoup(html_crudo, 'html.parser')
print(f"Tipo de 'sopa': {type(sopa)}")
# --- EXTRACCIÓN DE DATOS ---

# Buscamos todos los bloques de libros (artículos con clase "product_pod")
bloques_libros = sopa.find_all('article', class_='product_pod')

datos_extraidos = []

for libro in bloques_libros:
    # Extraemos el título (está dentro del atributo 'title' de la etiqueta <a> dentro de <h3>)
    titulo = libro.h3.a['title']
    print("titulo",titulo)
    # Extraemos el precio
    precio = libro.find('p', class_='price_color').text

    # Extraemos la disponibilidad (Stock)
    disponibilidad = libro.find('p', class_='instock availability').text.strip()

    # Extraemos la calificación (clase "star-rating ...")
    # Nota: La calificación es la segunda clase, ej: "star-rating Three"
    rating_clases = libro.find('p', class_='star-rating')['class']
    calificacion = rating_clases[1] # Esto nos dará "One", "Two", etc.

    # Guardamos como diccionario
    datos_extraidos.append({
        'Título': titulo,
        'Precio': precio,
        'Disponibilidad': disponibilidad,
        'Rating': calificacion
    })

# Convertimos a DataFrame para visualizar mejor
df_libros = pd.DataFrame(datos_extraidos)

print(f"Página 9 descargada con éxito. Se encontraron {len(df_libros)} libros.")
print(df_libros.head())

Tipo de 'html_crudo': <class 'str'>
Tipo de 'sopa': <class 'bs4.BeautifulSoup'>
titulo The Bridge to Consciousness: I'm Writing the Bridge Between Science and Our Old and New Beliefs.
titulo The Artist's Way: A Spiritual Path to Higher Creativity
titulo The Art of War
titulo The Argonauts
titulo The 10% Entrepreneur: Live Your Startup Dream Without Quitting Your Day Job
titulo Suddenly in Love (Lake Haven #1)
titulo Something More Than This
titulo Soft Apocalypse
titulo So You've Been Publicly Shamed
titulo Shoe Dog: A Memoir by the Creator of NIKE
titulo Shobu Samurai, Project Aryoku (#3)
titulo Secrets and Lace (Fatal Hearts #1)
titulo Scarlett Epstein Hates It Here
titulo Romero and Juliet: A Tragic Tale of Love and Zombies
titulo Redeeming Love
titulo Poses for Artists Volume 1 - Dynamic and Sitting Poses: An Essential Reference for Figure Drawing and the Human Form
titulo Poems That Make Grown Women Cry
titulo Nightingale, Sing
titulo Night Sky with Exit Wounds
titulo Mrs. Houdini

In [ ]:
# FASE 2: Web Scraping + Preprocesamiento
import requests
from bs4 import BeautifulSoup
import pandas as pd

# 1. URL
url_scrape = "http://books.toscrape.com/catalogue/page-9.html"

# 2. Request
respuesta_web = requests.get(url_scrape)
html_crudo = respuesta_web.text

# 3. Parseo
sopa = BeautifulSoup(html_crudo, 'html.parser')

bloques_libros = sopa.find_all('article', class_='product_pod')

datos_extraidos = []

for libro in bloques_libros:
    titulo = libro.h3.a['title']
    precio = libro.find('p', class_='price_color').text

    rating_clases = libro.find('p', class_='star-rating')['class']
    calificacion = rating_clases[1]

    datos_extraidos.append({
        'titulo': titulo,
        'precio_raw': precio,
        'rating_texto': calificacion
    })

# DataFrame
df_libros = pd.DataFrame(datos_extraidos)

# --- LIMPIEZA (clave para ETL) ---

# Normalizar texto
df_libros["titulo"] = df_libros["titulo"].str.strip().str.lower()

# Convertir precio
df_libros["precio_gbp"] = (
    df_libros["precio_raw"]
    .str.replace('[^0-9.]', '', regex=True)
    .astype(float)
)

# Mapear rating
map_rating = {
    "One": 1,
    "Two": 2,
    "Three": 3,
    "Four": 4,
    "Five": 5
}

df_libros["rating"] = df_libros["rating_texto"].map(map_rating)

# Dataset final de Fase 2
df_libros = df_libros[["titulo", "precio_gbp", "rating"]]

print(df_libros.head())

                                              titulo  precio_gbp  rating
0  the bridge to consciousness: i'm writing the b...       32.00       3
1  the artist's way: a spiritual path to higher c...       38.49       5
2                                     the art of war       33.34       5
3                                      the argonauts       10.93       2
4  the 10% entrepreneur: live your startup dream ...       27.55       3


In [ ]:
import requests

def obtener_tipo_cambio(debug=False):
    url = "https://open.er-api.com/v6/latest/GBP"

    try:
        if debug: print("[INFO] Llamando a la API...")

        r = requests.get(url, timeout=10)

        if debug: print(f"[INFO] Status: {r.status_code}")

        r.raise_for_status()

        data = r.json()

        if debug:
            print("[INFO] JSON recibido")
            print(f"[INFO] Base: {data.get('base_code')}")

        if data.get("result") != "success":
            raise ValueError("Respuesta inválida")

        tipo_cambio = data["rates"]["MXN"]

        if debug:
            print(f"[OK] Tipo de cambio obtenido: {tipo_cambio}")

        return float(tipo_cambio)

    except Exception as e:
        if debug:
            print(f"[ERROR] {e}")
            print("[WARN] Se usará fallback")

        return 23.0 # fallback

if __name__ == "__main__":
    tipo_cambio = obtener_tipo_cambio(debug=True)
    print(f"Tipo de cambio final: {tipo_cambio}")

[INFO] Llamando a la API...
[INFO] Status: 200
[INFO] JSON recibido
[INFO] Base: GBP
[OK] Tipo de cambio obtenido: 23.980932
Tipo de cambio final: 23.980932


In [ ]:
# Implementación completa de Fase 4 (ETL)
import pandas as pd
import sqlite3

# --- 4. LEER BASE DE DATOS SQL ---
conn = sqlite3.connect("tarifas.db")
# Changed from 'tarifas_importacion' to 'aduana' as per setup in cell 'yNBOspnW-_yZ' and 'ZchHbyEd3Ykl'
df_tarifas = pd.read_sql("SELECT * FROM aduana", conn)

# Verifica columnas esperadas:
# rating_id | tarifa_gbp
# Renaming for consistency with 'rating' column in df_libros
df_tarifas = df_tarifas.rename(columns={'rating_id': 'rating', 'tarifa_gbp': 'tarifa_importacion'})

# --- 5. LEER CSV ---
df_descuentos = pd.read_csv("descuentos.csv")

# --- 6. OBTENER TIPO DE CAMBIO ---
# The function 'obtener_tipo_cambio' is defined in cell QGk3KMgW9p39
tipo_cambio = obtener_tipo_cambio()

# --- 7. JOINS Y APLICACIÓN DE DESCUENTO ---
# Start with df_libros (from FASE 2) which already has 'titulo', 'precio_gbp', 'rating'
df = df_libros.copy()

# Join with SQL (by rating) to get import tariffs
df = df.merge(df_tarifas, on="rating", how="left")

# Apply global discount. Assuming df_descuentos provides a single factor_descuento.
# Given df_descuentos has no 'titulo' column, it's likely a global discount.
if not df_descuentos.empty:
    # Assuming the first factor_descuento is the one to apply globally
    global_discount_factor = df_descuentos['factor_descuento'].iloc[0]
else:
    global_discount_factor = 1.0 # No discount if CSV is empty

df['factor_descuento'] = global_discount_factor

# --- 8. MANEJO DE NULOS ---
# Fill any NaNs that might arise from left merges (e.g., if a rating has no tariff)
df["tarifa_importacion"] = df["tarifa_importacion"].fillna(0) # No import tariff if not found

# --- 9. CÁLCULO FINAL ---
# Precio con descuento aplicado, luego suma la tarifa de importación, y convierte a MXN
df["precio_mxn"] = (
    (df["precio_gbp"] * df["factor_descuento"] + df["tarifa_importacion"])
    * tipo_cambio
)

# --- 10. RESULTADO FINAL ---
df_final = df[[
    "titulo",
    "rating",
    "precio_gbp",
    "factor_descuento",
    "tarifa_importacion",
    "precio_mxn"
]]

print(df_final.head())

                                              titulo  rating  precio_gbp  \
0  the bridge to consciousness: i'm writing the b...       3       32.00   
1  the artist's way: a spiritual path to higher c...       5       38.49   
2                                     the art of war       5       33.34   
3                                      the argonauts       2       10.93   
4  the 10% entrepreneur: live your startup dream ...       3       27.55   

   factor_descuento  tarifa_importacion  precio_mxn  
0              0.85                 3.0  724.224146  
1              0.85                 1.0  808.553094  
2              0.85                 1.0  703.576564  
3              0.85                 4.0  318.718577  
4              0.85                 3.0  633.516271  


In [ ]:
# Fase 5: Exportaci ́on. Exporta tu DataFrame final fusionado y calculado a un archivo llamado tuBoleta dataset final.csv.

# Nombre del archivo (ajústalo con tu boleta)
nombre_archivo = "tuBoleta_dataset_final.csv"

# Exportar a CSV
df_final.to_csv(nombre_archivo, index=False, encoding="utf-8")

print(f"Archivo '{nombre_archivo}' exportado correctamente.")

df_verificacion = pd.read_csv(nombre_archivo)
print(df_verificacion.head())

Archivo 'tuBoleta_dataset_final.csv' exportado correctamente.
                                              titulo  rating  precio_gbp  \
0  the bridge to consciousness: i'm writing the b...       3       32.00   
1  the artist's way: a spiritual path to higher c...       5       38.49   
2                                     the art of war       5       33.34   
3                                      the argonauts       2       10.93   
4  the 10% entrepreneur: live your startup dream ...       3       27.55   

   factor_descuento  tarifa_importacion  precio_mxn  
0              0.85                 3.0  724.224146  
1              0.85                 1.0  808.553094  
2              0.85                 1.0  703.576564  
3              0.85                 4.0  318.718577  
4              0.85                 3.0  633.516271  


In [ ]:
# Fase 5: Exportaci ́on. Exporta tu DataFrame final fusionado y calculado a un archivo llamado tuBoleta dataset final.csv.

# Nombre del archivo (ajústalo con tu boleta)
nombre_archivo = "Mejia_Reyes_dataset_final.csv"

# Exportar a CSV
df_final.to_csv(nombre_archivo, index=False, encoding="utf-8")

print(f"Archivo '{nombre_archivo}' exportado correctamente.")

df_verificacion = pd.read_csv(nombre_archivo)
print(df_verificacion.head())

Archivo 'Mejia_Reyes_dataset_final.csv' exportado correctamente.
                                              titulo  rating  precio_gbp  \
0  the bridge to consciousness: i'm writing the b...       3       32.00   
1  the artist's way: a spiritual path to higher c...       5       38.49   
2                                     the art of war       5       33.34   
3                                      the argonauts       2       10.93   
4  the 10% entrepreneur: live your startup dream ...       3       27.55   

   factor_descuento  tarifa_importacion  precio_mxn  
0              0.85                 3.0  724.224146  
1              0.85                 1.0  808.553094  
2              0.85                 1.0  703.576564  
3              0.85                 4.0  318.718577  
4              0.85                 3.0  633.516271  
